# 일자·시간대 매출 자료 확보와 적용 가능성

**확인일: 2026-10-05 · 졸업프로젝트 자료 확보 기록**

목표는 종로구·마포구의 2023~2025년 공연 종료 후 0~3시간·3~6시간 매출을 관측할 수 있는 자료를 찾는 것이다. 통계적 유의성이 높아질 자료를 선택하는 작업이 아니라, 연구 질문에 맞는 관측 단위를 확보하는 작업이다.

**결론:** 현재 기간을 포함하는 B079는 날짜별이지만 6개 고정 시간 구간이다. 1시간 단위 B042는 2022년까지여서 현재 기간과 맞지 않는다. 둘 모두 캠퍼스 내부 분석 자료다. 이 노트북은 공개 설명서와 안내 페이지를 수집하고, 공개 가공 샘플도 별도로 수집하되, 이를 매출 원본으로 간주하지 않는다.

실행 기본값은 저장한 원문의 **오프라인 검증**이다. 재수집은 환경변수 `TIME_SALES_FETCH=1`로 실행하며 매번 새 원문 폴더를 만든다. 기존 분석 입력·회귀 결과는 변경하지 않는다.

## 1. 저장 위치와 출처

수집 원문은 새 회차에 보관하고 SHA-256과 조회 시각을 기록한다. 이 명세는 자료 접근성 확인용이며 기존 매출 분석의 입력 명세와 구분한다.

In [1]:
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
import os, json, hashlib, uuid, time
import pandas as pd
from IPython.display import display
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'AGENTS.md').exists() and (p / 'final').is_dir())
META = ROOT / 'final/metadata/time_sales_access_sources.json'
OUT = ROOT / 'reports/final/data_access'
OUT.mkdir(parents=True, exist_ok=True)
FETCH = os.environ.get('TIME_SALES_FETCH', '0') == '1'
NOW = lambda: datetime.now(ZoneInfo('Asia/Seoul')).isoformat(timespec='seconds')
SOURCES = [
    ('b079_catalog.html', 'html', 'https://bigdata.seoul.go.kr/data/selectSampleData.do?sample_data_seq=333'),
    ('b042_catalog.html', 'html', 'https://bigdata.seoul.go.kr/data/selectSampleData.do?r_id=P213&sample_data_seq=51'),
    ('campus_data_dictionary.pdf', 'pdf', 'https://bigdata.seoul.go.kr/common/file/directFileDown.do?ad=data&file_nm=빅데이터_캠퍼스_데이터_설명서.pdf'),
    ('campus_export_policy.pdf', 'pdf', 'https://bigdata.seoul.go.kr/common/file/directFileDown.do?ad=tkout&file_nm=빅데이터_캠퍼스_반출_정책.pdf'),
    ('campus_use_guide.html', 'html', 'https://bigdata.seoul.go.kr/use/useInfo.do?r_id=P340'),
    ('campus_data_use.html', 'html', 'https://bigdata.seoul.go.kr/cnts.do?r_id=P212'),
]
print('모드:', '새 회차 수집' if FETCH else '고정 원문 오프라인 검증')

모드: 고정 원문 오프라인 검증


## 2. 공식 문서 수집 또는 고정본 검증

로그인·차단 안내가 나오면 우회하지 않는다. PDF는 파일 시그니처를 확인해 오류 HTML이 설명서로 저장되는 일을 방지한다. 공개 HTML은 안내 내용과 샘플 버튼의 로그인 요구 여부를 확인하는 근거다.

In [2]:
def digest(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

if FETCH:
    import requests
    run_id = datetime.now(ZoneInfo('Asia/Seoul')).strftime('%Y%m%d_%H%M%S') + '_' + uuid.uuid4().hex[:6]
    raw_dir = ROOT / 'data/raw/time_sales_access' / run_id
    raw_dir.mkdir(parents=True, exist_ok=False)
    manifest = {'run_id': run_id, 'created_at': NOW(), 'purpose': 'public_documentation_and_processed_samples',
                'sales_microdata_downloaded': False, 'public_sample_downloaded': False, 'files': []}
    session = requests.Session()
    try:
        for filename, kind, url in SOURCES:
            response = session.get(url, timeout=(15, 120))
            response.raise_for_status()
            content = response.content
            content_type = response.headers.get('Content-Type', '')
            if kind == 'pdf' and not content.startswith(b'%PDF-'):
                raise RuntimeError('PDF 대신 다른 응답을 받음: ' + filename)
            if kind == 'html':
                html_text = content.decode('utf-8', errors='replace')
                from urllib.parse import urlsplit
                expected_text = {'b079_catalog.html': '서울시민의 업종별 카드소비 데이터',
                                 'b042_catalog.html': '서울시 업종별 내외국인 신한카드 매출데이터',
                                 'campus_use_guide.html': '캠퍼스 이용신청 항목은 모두 필수 입력 사항',
                                 'campus_data_use.html': '원본파일서버'}[filename]
                same_page = urlsplit(response.url).path == urlsplit(url).path
                if not same_page or expected_text not in html_text or '<html' not in html_text.lower():
                    raise RuntimeError('예상 페이지와 다름: 로그인·차단 응답 가능성, 요청 중단: ' + filename)
            path = raw_dir / filename
            path.write_bytes(content)
            manifest['files'].append({'path': path.relative_to(ROOT).as_posix(), 'kind': kind,
                'source_url': url, 'resolved_url': response.url, 'retrieved_at': NOW(),
                'http_status': response.status_code, 'content_type': content_type,
                'bytes': len(content), 'sha256': digest(path)})
            time.sleep(1)
        from bs4 import BeautifulSoup
        from urllib.parse import urljoin
        import re
        sample_plan = [
            ('b079_catalog.html', 4, 'B079_05_domestic_time_PUBLIC_SAMPLE.csv'),
            ('b079_catalog.html', 8, 'B079_09_foreign_time_PUBLIC_SAMPLE.csv'),
            ('b042_catalog.html', 0, 'B042_domestic_hour_PUBLIC_SAMPLE.csv'),
            ('b042_catalog.html', 5, 'B042_domestic_industry_PUBLIC_SAMPLE.csv'),
        ]
        for catalog_name, index, filename in sample_plan:
            catalog_text = (raw_dir / catalog_name).read_text(encoding='utf-8')
            soup = BeautifulSoup(catalog_text, 'html.parser')
            form = soup.find('form', id='aform')
            assert form.get('method', 'get').lower() == 'get'
            anchors = [a for a in soup.find_all('a') if '샘플데이터 다운로드' in a.get_text()]
            anchor = anchors[index]
            file_id = re.search(r"fnDownload\('([^']+)'\)", anchor.get('onclick', '')).group(1)
            # 관측한 공개 페이지의 다운로드 함수와 동일한 주소만 사용한다.
            assert "'action' : '/common/file/FileDown.do'" in catalog_text
            params = {i['name']: i.get('value', '') for i in form.find_all('input') if i.get('name')}
            params['file_id'] = file_id
            source_url = next(x['source_url'] for x in manifest['files'] if Path(x['path']).name == catalog_name)
            response = session.get(urljoin(source_url, '/common/file/FileDown.do'), params=params,
                                   headers={'Referer': source_url}, timeout=(15, 60))
            response.raise_for_status()
            content = response.content
            if 'text/html' in response.headers.get('Content-Type', '').lower() or b'<html' in content[:1000].lower():
                raise RuntimeError('샘플 대신 안내/로그인 응답: 요청 중단')
            path = raw_dir / filename
            path.write_bytes(content)
            manifest['files'].append({'path': path.relative_to(ROOT).as_posix(),
                'kind': 'public_processed_sample_csv', 'source_url': source_url,
                'resolved_url': response.url, 'source_label': anchor.find_parent('td').get_text(' ', strip=True),
                'retrieved_at': NOW(), 'http_status': response.status_code,
                'content_type': response.headers.get('Content-Type', ''),
                'bytes': len(content), 'sha256': digest(path)})
            time.sleep(1)
        manifest['public_sample_downloaded'] = True
        manifest['status'] = 'complete_documentation_and_public_samples'
    except Exception as exc:
        manifest['status'] = 'stopped'
        manifest['error_type'] = type(exc).__name__
        raise
    finally:
        (raw_dir / 'manifest.json').write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')
        session.close()
    META.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')
else:
    if not META.exists():
        raise FileNotFoundError('최초 수집이 필요합니다: TIME_SALES_FETCH=1')
    manifest = json.loads(META.read_text(encoding='utf-8'))

checks = []
for item in manifest['files']:
    path = ROOT / item['path']
    valid = path.is_file() and digest(path) == item['sha256']
    checks.append({'파일': path.name, '크기_bytes': item['bytes'], 'SHA256_일치': valid, '종류': item['kind']})
assert sum(x['종류'] in ['html', 'pdf'] for x in checks) == len(SOURCES)
assert sum(x['종류'] == 'public_processed_sample_csv' for x in checks) == 4
assert all(x['SHA256_일치'] for x in checks)
display(pd.DataFrame(checks))

,파일,크기_bytes,SHA256_일치,종류
0,b079_catalog.html,494079,True,html
1,b042_catalog.html,816887,True,html
2,campus_data_dictionary.pdf,2592426,True,pdf
3,campus_export_policy.pdf,327512,True,pdf
4,campus_use_guide.html,25800,True,html
5,campus_data_use.html,20186,True,html
6,B079_05_domestic_time_PUBLIC_SAMPLE.csv,10022,True,public_processed_sample_csv
7,B079_09_foreign_time_PUBLIC_SAMPLE.csv,8364,True,public_processed_sample_csv
8,B042_domestic_hour_PUBLIC_SAMPLE.csv,32192,True,public_processed_sample_csv
9,B042_domestic_industry_PUBLIC_SAMPLE.csv,2948,True,public_processed_sample_csv


**해석:** 위 목록은 공식 문서·안내 페이지·공개 가공 샘플의 다운로드 결과다. 매출 원본 파일의 존재·관측 건수·결측률을 검증한 표가 아니다.

## 3. 샘플 다운로드 경로 확인

비로그인 공개 페이지에서 관측되는 다운로드 버튼만 검사한다. 샘플은 원본 이해용 가공 자료이며, 확보하더라도 실제 연구 표본으로 사용하지 않는다.

In [3]:
from bs4 import BeautifulSoup
files = {Path(x['path']).name: ROOT / x['path'] for x in manifest['files']}
sample_links = []
for catalog in ['b079_catalog.html', 'b042_catalog.html']:
    soup = BeautifulSoup(files[catalog].read_text(encoding='utf-8'), 'html.parser')
    for anchor in soup.find_all('a'):
        if '샘플데이터 다운로드' in anchor.get_text(' ', strip=True):
            sample_links.append({'목록': catalog, '버튼': anchor.get_text(' ', strip=True),
                                 'href': anchor.get('href', ''), 'onclick': anchor.get('onclick', '')})
links_df = pd.DataFrame(sample_links)
links_df.to_csv(OUT / 'sample_download_links.csv', index=False, encoding='utf-8-sig')
display(links_df.groupby('목록', as_index=False).size().rename(columns={'size': '공개 샘플 버튼 수'}))
print('매출 원본 확보:', manifest['sales_microdata_downloaded'])
print('샘플 확보:', manifest['public_sample_downloaded'])

,목록,공개 샘플 버튼 수
0,b042_catalog.html,7
1,b079_catalog.html,13


매출 원본 확보: False
샘플 확보: True


### 공개 가공 샘플의 구조 검증

아래 표는 샘플의 실제 열, 날짜 범위, 시간코드와 업종 범위를 확인한다. 작은 예시 자료에서 전체 모집단의 완전성이나 결측률을 추정하지 않는다. 프로파일링 결과도 기존 회귀분석에 투입하지 않는다.

In [4]:
profiles = []
sample_frames = {}
for item in manifest['files']:
    if item['kind'] != 'public_processed_sample_csv':
        continue
    path = ROOT / item['path']
    frame = None
    for encoding in ['utf-8-sig', 'cp949']:
        try:
            frame = pd.read_csv(path, encoding=encoding, dtype=str, keep_default_na=False)
            break
        except UnicodeDecodeError:
            continue
    if frame is None:
        raise ValueError('확인되지 않은 샘플 인코딩: ' + path.name)
    sample_frames[path.name] = frame
    assert len(frame.columns) > 1 and len(frame) > 0
    date_col = next((c for c in frame.columns if c == '기준일자' or c.endswith('(TS_YMD)')), None)
    time_col = next((c for c in frame.columns if c == '시간대' or c.endswith('(TM)')), None)
    profiles.append({'파일': path.name, '예시행수': len(frame), '열': ', '.join(frame.columns),
                     '예시날짜': (frame[date_col].min() + '~' + frame[date_col].max()) if date_col else '해당 없음',
                     '예시시간코드': ', '.join(sorted(frame[time_col].unique())) if time_col else '해당 없음'})
profile = pd.DataFrame(profiles)
profile.to_csv(OUT / 'public_sample_schema.csv', index=False, encoding='utf-8-sig')
with pd.option_context('display.max_colwidth', 180):
    display(profile)

public_industries = sample_frames['B079_05_domestic_time_PUBLIC_SAMPLE.csv']['업종대분류'].drop_duplicates().sort_values().tolist()
print('B079 내국인 공개 샘플의 업종값:', ', '.join(public_industries))
print('샘플에 등장하지 않는 업종이 전체 데이터에 없다는 뜻은 아닙니다.')


,파일,예시행수,열,예시날짜,예시시간코드
0,B079_05_domestic_time_PUBLIC_SAMPLE.csv,200,"기준일자, 요일, 시간대, 격자_50, 업종대분류, 카드이용금액계, 카드이용건수계",20230101~20230101,"1, 2, 3, 4, 5, 6"
1,B079_09_foreign_time_PUBLIC_SAMPLE.csv,200,"기준일자, 시간대, 격자_50, 업종대분류, 카드이용금액계, 카드이용건수계",20230101~20230101,"1, 2, 3, 4, 5, 6"
2,B042_domestic_hour_PUBLIC_SAMPLE.csv,500,"가맹점블록코드(BLCK_CD), 내국인업종코드(SB_UPJONG_CD), 기준년월(TS_YM), 일별(TS_YMD), 요일(DAW), 시간대(TM), 카드이용금액계(AMT_CORR), 카드이용건수(USECT_CORR)",20170101~20210729,"00, 01, 02, 03, 04, 05, 06, 07, 08, 09, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23"
3,B042_domestic_industry_PUBLIC_SAMPLE.csv,63,"대분류(SB_L_UPJONG_NM), 중분류(SB_M_UPJONG_NM), 내국인업종분류(SB_UPJONG_NM), 내국인업종코드(SB_UPJONG_CD)",해당 없음,해당 없음


B079 내국인 공개 샘플의 업종값: ZZ_나머지, 가전, 게임방/오락실, 결제대행(PG), 기타식품, 기타요식, 기타유통, 노래방, 농수산물, 대형마트, 독서실, 동물병원, 모텔,여관,기타숙박, 미용실, 생활잡화/수입상품점, 세탁소, 슈퍼마켓일반형, 스포츠/레저용품, 스포츠시설, 실내/실외골프장, 약국, 양식, 의복/의류, 일식, 정육점, 제과점, 주유소, 중식, 커피전문점, 패스트푸드, 편의점, 한식, 한의원, 화원
샘플에 등장하지 않는 업종이 전체 데이터에 없다는 뜻은 아닙니다.


**샘플에서 직접 확인한 내용:** B079 내국인·외국인 파일은 각각 200행이며 날짜·시간구간·가맹점 격자·업종·금액·건수가 같은 행에 있다. 예시 날짜는 2023-01-01이고, 내국인 업종값에는 한식·중식·일식·양식·커피전문점·제과점·패스트푸드가 포함된다. 파일의 “대분류”라는 명칭만으로 음식점 전체가 하나로 합쳐졌다고 단정할 수 없다. 다만 치킨·분식·주점 등을 포함한 기존 10종의 완전한 대응은 코드북 확인이 필요하다.

B042 매출 샘플은 500행, 업종 코드 샘플은 63행이며 시간코드 00~23을 확인했다. **샘플의 날짜 범위는 전체 제공기간이 아니며, B079의 2023년 50m 예시도 모든 월의 같은 격자 버전을 보장하지 않는다.**

## 4. 자료별로 가능한 분석

자료 이름의 “시간대별”만으로 시간 해상도를 판단하지 않는다. 날짜·시간·판매 위치·업종·금액이 같은 행에 있어야 한다. B079는 공식 데이터 설명서 184쪽의 **05번 내국인 일자별·시간대별 격자 자료**를 우선 검토한다. 02번의 고객 거주 행정동을 공연장 주변 판매지로 바꾸어 해석해서는 안 된다.

In [5]:
candidates = pd.DataFrame([
    {'자료': 'B079 / 내국인 05', '공식기간': '2021.01~2026.07', '날짜와 시간': '일자 × 6개 고정 구간',
     '공간': '가맹점 격자; 250m/50m 버전 확인 필요', '2023~25': '포함',
     '적합성': '일별 고정 시간대 비교 가능; 정확한 종료 후 3/6시간 매출은 불가',
     '접근': '이용신청·방문·내부 분석·결과 반출심사',
     '출처': SOURCES[0][2]},
    {'자료': 'B042 / 내국인 블록', '공식기간': '2017.01~2022.12', '날짜와 시간': '일자 × 1시간',
     '공간': '가맹점 블록', '2023~25': '미포함',
     '적합성': '과거 기간 별도 설계가 필요; 임의 분 단위 종료시각에는 부분시간 가정 필요',
     '접근': '이용신청·방문·내부 분석·결과 반출심사',
     '출처': SOURCES[1][2]},
    {'자료': '서울 실시간 상권현황', '공식기간': '실시간', '날짜와 시간': '최근 10분',
     '공간': '지정 주요 장소', '2023~25': '과거 제공 불가',
     '적합성': '현재 연구 소급 분석 불가', '접근': '공개 API',
     '출처': 'https://data.seoul.go.kr/dataVisual/seoul/guide.do'},
    {'자료': '삼성카드 일별 시간대별 소비트렌드', '공식기간': '현재 공급기간 미확인', '날짜와 시간': '일자·시간대 열 있음',
     '공간': '공개 샘플 열에는 없음; 지역 세분화 요청 사항', '2023~25': '미확인',
     '적합성': '공간·기간·시간 코드·제공 조건 확인 전 사용 불가', '접근': '협의 상품',
     '출처': 'https://www.findatamall.or.kr/market/dataProdDetail?gdsSn=116&menuNo=0'},
])
candidates.to_csv(OUT / 'candidate_datasets.csv', index=False, encoding='utf-8-sig')
display(candidates.drop(columns='출처'))

,자료,공식기간,날짜와 시간,공간,2023~25,적합성,접근
0,B079 / 내국인 05,2021.01~2026.07,일자 × 6개 고정 구간,가맹점 격자; 250m/50m 버전 확인 필요,포함,일별 고정 시간대 비교 가능; 정확한 종료 후 3/6시간 매출은 불가,이용신청·방문·내부 분석·결과 반출심사
1,B042 / 내국인 블록,2017.01~2022.12,일자 × 1시간,가맹점 블록,미포함,과거 기간 별도 설계가 필요; 임의 분 단위 종료시각에는 부분시간 가정 필요,이용신청·방문·내부 분석·결과 반출심사
2,서울 실시간 상권현황,실시간,최근 10분,지정 주요 장소,과거 제공 불가,현재 연구 소급 분석 불가,공개 API
3,삼성카드 일별 시간대별 소비트렌드,현재 공급기간 미확인,일자·시간대 열 있음,공개 샘플 열에는 없음; 지역 세분화 요청 사항,미확인,공간·기간·시간 코드·제공 조건 확인 전 사용 불가,협의 상품


**해석:** 확인한 공식 경로에서는 현재 연구 조건에 맞는 무료 다운로드형 정밀 매출 원본을 찾지 못했다. “2023~2025년 포함”과 “공연 종료 직후를 측정 가능”은 서로 다른 조건이다. [서울 실시간 자료의 과거 제공 제한](https://data.seoul.go.kr/dataVisual/seoul/guide.do)과 [삼성카드 상품의 별도 협의 조건](https://www.findatamall.or.kr/market/dataProdDetail?gdsSn=116&menuNo=0)도 확인했다.

B079의 시간 구간은 **00~06 / 06~11 / 11~14 / 14~17 / 17~21 / 21~24시**다. 예를 들어 20시 30분 종료 공연의 3시간 후는 23시 30분이다. 이를 17~21시와 21~24시 매출에서 잘라 계산하면 구간 안의 매출이 균등하다는 확인되지 않은 가정이 들어간다. 이 값은 관측 매출이 아니다.

[공식 데이터 설명서](https://bigdata.seoul.go.kr/common/file/directFileDown.do?ad=data&file_nm=빅데이터_캠퍼스_데이터_설명서.pdf)의 B079 182~186쪽, B042 68~70쪽을 기준으로 확인했다. 설명서에서 업종 항목은 대분류로 표기되어 있으므로 기존 외식업 10종과 일대일 연결되는지는 별도 코드북 확인이 필요하다.

## 5. 신청 전에 확인할 항목

1. **시간:** B079에 1시간·30분 또는 더 세밀한 별도 자료가 있는가? 없다면 정확한 종료 후 시간창 매출 확보는 미충족으로 기록하고, 일별 6개 고정 구간 분석의 가능 범위를 별도로 제시한다.
2. **공간:** 소개의 “2025년 7월부터 50m”와 상세 설명의 “2023년부터 50m”가 과거 재작성 여부를 뜻하는가? 2023~2025년에 공통으로 적용 가능한 경계와 CRS, 격자 대응표가 있는가?
3. **업종:** 05번 데이터의 업종 코드북에서 기존 외식업 10종을 구별할 수 있는가? 대분류만 있다면 임의로 세분화하지 않는다.
4. **금액:** 원 카드 이용액인지 보정 추정액인지, 제외 카드·미관측·소액 비공개·취소 거래의 처리 기준은 무엇인가?
5. **이용:** KOPIS 공연 시간·좌표 자료 및 오프라인 분석 노트북을 반입하고 회귀 결과·검정·보고서를 반출할 수 있는가?

미관측 행은 매출 0원이 아니다. 다른 카드 모집단이나 경계 버전을 기존 매출과 단순 접합하지 않는다.

## 6. 이용신청에 붙여 넣을 연구 설명 초안

**제목:** 공연 활동과 주변 외식업의 일별 시간대별 소비 관계 분석 — 학부 졸업프로젝트

**신청자:** [이름] / 빅데이터응용학과 / [학번] / [이메일] / [전화번호]  
**지도교수:** 김태경  
**희망 이용 기간:** [방문 가능한 날짜 기입]

**이용 목적:** 종로구·마포구의 공연 일정 및 공연장 위치를 주변 외식업의 날짜별 시간대별 카드 이용액과 연결하여, 공연이 있는 날과 비교 가능한 날의 소비 양상을 분석하고자 합니다. 기존 분기 자료로는 공연 당일의 시간별 변화를 확인하기 어려워 세부 자료 이용을 신청합니다. 단순한 지역 간 매출 차이와 공연 관련 변화를 구분하고, 통계적으로 유의하지 않은 결과도 함께 보고하겠습니다.

**희망 자료:** B079의 05번 서울시 내국인 일자별·시간대별 격자 자료 중 2023-01-01~2025-12-31, 종로구·마포구에 해당하는 가맹점 격자와 외식업 코드. 해당 기간의 업종 코드북, 격자 경계·좌표계·버전 대응표, 금액·건수·비공개 처리 설명이 필요합니다. 외국인 자료는 별도 모집단의 보조 분석에 필요한 경우에만 검토합니다.

**분석 계획:** 시간 해상도가 6구간이면 종료 후 정확한 3시간 매출로 해석하지 않고 일별 고정 시간 구간 분석으로 진행합니다. 가맹점 격자와 공연장 거리, 날짜별 공연 회차 정보를 결합하고 지역·업종·요일·날짜·시간대 차이와 공휴일을 고려합니다. KOPIS의 안내 일정이 실제 회차별 이력과 일치하는지 먼저 점검합니다. 인접 공연의 중첩과 비교 지역의 공연 노출을 확인한 뒤 모형과 평가 구간을 정합니다.

**반입·반출 계획:** 공개 KOPIS 자료와 인증키를 제거한 오프라인 노트북을 사전 반입 신청합니다. 원본·발췌·일별 시간별 매출표의 외부 반출을 전제로 하지 않으며, 캠퍼스 내부에서 분석을 완료하고 회귀 계수·신뢰구간·검정 결과·보고서와 심사 가능한 그림의 반출을 신청합니다. 원자료 행을 출력한 노트북을 제출하지 않겠습니다.

## 7. 담당자에게 확인할 문의 초안 — 미발송

안녕하세요. 빅데이터응용학과 학부 졸업프로젝트를 준비하고 있습니다. 종로구·마포구의 2023~2025년 공연과 주변 외식업 소비 관계 분석에 B079 05번 자료를 활용하고자 합니다.

설명서의 6개 시간 구간보다 세밀한 1시간·30분 단위 자료가 별도로 제공되는지, 50m 격자 자료가 2023년까지 소급 제공되는지 확인 부탁드립니다. 또한 외식업 10종을 구별할 업종 코드북과 격자 경계 자료를 함께 이용할 수 있는지 궁금합니다. KOPIS 공개자료와 오프라인 노트북의 반입 및 회귀 결과·검정·보고서의 반출 가능 범위도 안내 부탁드립니다.

[이름] / [연락처]

※ 이 노트북은 신청·문의 문안을 준비한 것이며 제출이나 발송을 수행하지 않는다.

## 8. 실제 다음 단계

[공식 이용가이드](https://bigdata.seoul.go.kr/use/useInfo.do?r_id=P340)에 따라 서울시 개인회원 로그인 → 방문일을 정해 이용신청 → 승인 후 캠퍼스 방문 → 내부 분석 → 결과 반출심사를 거친다. 신청은 최대 3일이고 연장할 수 있다. [자료 이용 안내](https://bigdata.seoul.go.kr/cnts.do?r_id=P212)도 원본 이용을 캠퍼스 내부 기준으로 설명한다.

[2026년 5월 반출정책](https://bigdata.seoul.go.kr/common/file/directFileDown.do?ad=tkout&file_nm=빅데이터_캠퍼스_반출_정책.pdf)에서 B079는 원본·발췌·단순집계 반출 불가, 응용집계·시각화는 심사 대상이다. 따라서 이 자료의 매출 패널을 그대로 로컬 Agentic AI 서비스에 넣는 구성을 전제로 해서는 안 된다. 서비스는 기존 공개 분기 자료로 유지하고, 허용된 분석 결과만 연결하는 방안을 검토한다.

**연구 방향 제안:** 기간을 2023~2025년으로 유지한다면 B079 이용 가능 여부를 먼저 확인한다. 정확한 3/6시간 분석이 꼭 필요하다면 더 세밀한 자료 공급 조건을 별도로 확인해야 한다. B042를 쓰기 위해 기간을 과거로 바꾸는 결정은 2022년 등의 KOPIS 자료 확보와 당시 방역·영업시간 조건 검토까지 필요하므로 이번 작업에서 임의로 변경하지 않았다.

In [6]:
status = {'checked_at': NOW(), 'run_id': manifest['run_id'],
          'downloaded_document_count': sum(x['kind'] in ['pdf', 'html'] for x in manifest['files']),
          'downloaded_public_sample_count': sum(x['kind'] == 'public_processed_sample_csv' for x in manifest['files']), 'sales_data_downloaded': False,
          'sample_data_downloaded': manifest['public_sample_downloaded'], 'application_submitted': False, 'inquiry_sent': False,
          'current_analysis_input_changed': False,
          'next_dependency': '개인회원 로그인·방문 일정·자료 제공 단위 확인'}
(OUT / 'access_status.json').write_text(json.dumps(status, ensure_ascii=False, indent=2), encoding='utf-8')
display(pd.DataFrame([status]).T.rename(columns={0: '상태'}))
import nbformat
source_notebook = nbformat.read(ROOT / 'final/collection/05_일자시간대매출_자료확보.ipynb', as_version=4)
for prefix, filename in [('## 6.', '캠퍼스_이용신청_연구설명_초안.txt'), ('## 7.', '캠퍼스_자료문의_초안_미발송.txt')]:
    section = next(c.source for c in source_notebook.cells if c.cell_type == 'markdown' and c.source.startswith(prefix))
    (OUT / filename).write_text(section, encoding='utf-8')


,상태
checked_at,2026-10-05T05:31:20+09:00
run_id,20261005_052911_f98e59
downloaded_document_count,6
downloaded_public_sample_count,4
sales_data_downloaded,False
sample_data_downloaded,True
application_submitted,False
inquiry_sent,False
current_analysis_input_changed,False
next_dependency,개인회원 로그인·방문 일정·자료 제공 단위 확인
